<a href="https://colab.research.google.com/github/OfirW3/IIoT-Network-Intrusion-Detection-Technion-ML-Course-Final-Project/blob/main/notebooks/merge_and_clean_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Creating the dataset by uniting the datasets of the benign samples and attack samples and remove a fraqtion of values from each attack label and benign label


Output dataset: cic_iiot_2025_combined


In [1]:
# ==========================================
# CELL 1: SETUP & CONFIGURATION
# ==========================================
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime

# ================= CONFIG =================
ATTACK_FILE = "/content/drive/MyDrive/CIC_IIOT_2025/attack_samples_1sec.csv"
BENIGN_FILE = "/content/drive/MyDrive/CIC_IIOT_2025/benign_samples_1sec.csv"
OUT_DIR = "/content/drive/MyDrive/CIC_IIOT_2025"
FINAL_OUT_FILE = "cic_iiot_2025_cleaned_and_balanced.csv"

RANDOM_STATE = 42
BENIGN_FRAC = 0.10         # Adjust this to keep more/less benign traffic
MIN_COUNT_FOR_LABEL2 = 5   # Threshold for grouping rare attacks into "other"
# ==========================================

In [2]:
# ==========================================
# CELL 2: UTILITY & CLEANING FUNCTIONS
# ==========================================
def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def read_csv_simple(path):
    print(f"[{now()}] Reading: {path}")
    try:
        df = pd.read_csv(path, engine="c", low_memory=False, encoding="utf-8", on_bad_lines="skip")
        print(f"  -> Loaded {len(df):,} rows, {df.shape[1]} cols (C engine)")
        return df
    except Exception as e_c:
        print(f"  C engine failed, trying python engine: {repr(e_c)}")
        df = pd.read_csv(path, engine="python", encoding="utf-8", on_bad_lines="skip")
        print(f"  -> Loaded {len(df):,} rows, {df.shape[1]} cols (python engine)")
        return df

def canonicalize_label1(x):
    if pd.isna(x):
        return "attack"
    s = str(x).lower()
    if "benign" in s or s == "0":
        return "benign"
    return "attack"

def canonicalize_label2(x):
    if pd.isna(x):
        return "other"

    s = str(x).lower()
    s = re.sub(r"[\[\]\"']", "", s).strip()

    if s in ("none", "benign", "normal"):
        return "none"
    if any(k in s for k in ("ddos", "dos", "flood")):
        return "ddos"
    if any(k in s for k in ("scan", "recon", "nmap")):
        return "recon"
    if any(k in s for k in ("brute", "password")):
        return "brute"
    if any(k in s for k in ("malware", "trojan", "botnet", "ransom")):
        return "malware"
    if any(k in s for k in ("mitm", "man-in-the-middle")):
        return "mitm"

    if re.fullmatch(r"[-+]?\d+(\.\d+)?", s):
        return "other"

    return s if len(s) < 30 else "other"

def report_counts(before_counts, after_counts, title="Sampling Report"):
    print("\n" + "="*64)
    print(title)
    print("="*64)
    print(f"{'Attack Type':30s} | {'Before':>12s} | {'After':>12s}")
    print("-" * 64)
    keys = sorted(set(before_counts.keys()).union(after_counts.keys()))
    for k in keys:
        print(f"{str(k)[:30]:30s} | {before_counts.get(k,0):12d} | {after_counts.get(k,0):12d}")
    print("-" * 64 + "\n")

In [3]:
# ==========================================
# CELL 3: DATA LOADING & MERGING
# ==========================================
if not os.path.exists(ATTACK_FILE): raise FileNotFoundError(f"Missing: {ATTACK_FILE}")
if not os.path.exists(BENIGN_FILE): raise FileNotFoundError(f"Missing: {BENIGN_FILE}")

# Load datasets
df_attack = read_csv_simple(ATTACK_FILE)
df_benign = read_csv_simple(BENIGN_FILE)

# Ensure baseline labels exist before cleaning to prevent KeyErrors
for df, default_lbl in [(df_attack, "attack"), (df_benign, "benign")]:
    if "label1" not in df.columns:
        df["label1"] = default_lbl
    if "label2" not in df.columns:
        # Fallback to a generic 'label' column if label2 is missing
        fallback = "label" if "label" in df.columns else "label_full"
        df["label2"] = df[fallback] if fallback in df.columns else default_lbl

# Combine into one master dataframe for uniform cleaning
df_combined = pd.concat([df_attack, df_benign], ignore_index=True, sort=False)
print(f"[{now()}] Master dataset created: {df_combined.shape[0]:,} rows × {df_combined.shape[1]} cols")

# Free up memory
del df_attack, df_benign

[2026-03-11 13:53:28] Reading: /content/drive/MyDrive/CIC_IIOT_2025/attack_samples_1sec.csv
  -> Loaded 90,391 rows, 94 cols (C engine)
[2026-03-11 13:53:49] Reading: /content/drive/MyDrive/CIC_IIOT_2025/benign_samples_1sec.csv
  -> Loaded 136,800 rows, 94 cols (C engine)
[2026-03-11 13:53:53] Master dataset created: 227,191 rows × 94 cols


In [4]:
# ==========================================
# CELL 4: DATA CLEANING & PREPROCESSING
# ==========================================
print(f"[{now()}] Cleaning label1...")
df_combined["label1"] = df_combined["label1"].apply(canonicalize_label1)

print(f"[{now()}] Cleaning label2...")
df_combined["label2"] = df_combined["label2"].apply(canonicalize_label2)

# Ensure benign rows must have label2 = benign
df_combined.loc[df_combined["label1"] == "benign", "label2"] = "benign"

print(f"[{now()}] Filling numeric NaNs with medians...")
for col in df_combined.select_dtypes(include=np.number).columns:
    df_combined[col] = df_combined[col].fillna(df_combined[col].median())

print(f"[{now()}] Dropping constant columns...")
constant_cols = [
    c for c in df_combined.columns
    if c not in ("label1", "label2") and df_combined[c].nunique() <= 1
]
df_combined.drop(columns=constant_cols, inplace=True)
print(f"[{now()}] Dropped {len(constant_cols)} constant columns.")

print(f"[{now()}] Grouping rare label2 classes...")
counts = df_combined["label2"].value_counts()
rare = counts[counts < MIN_COUNT_FOR_LABEL2].index
df_combined.loc[df_combined["label2"].isin(rare), "label2"] = "other"
print(f"[{now()}] Grouped {len(rare)} rare classes into 'other'.")

print(f"[{now()}] Cleaning complete. Current shape: {df_combined.shape}")

[2026-03-11 13:53:53] Cleaning label1...
[2026-03-11 13:53:53] Cleaning label2...
[2026-03-11 13:53:53] Filling numeric NaNs with medians...
[2026-03-11 13:53:54] Dropping constant columns...
[2026-03-11 13:53:56] Dropped 0 constant columns.
[2026-03-11 13:53:56] Grouping rare label2 classes...
[2026-03-11 13:53:56] Grouped 0 rare classes into 'other'.
[2026-03-11 13:53:56] Cleaning complete. Current shape: (227191, 94)


In [5]:
# ==========================================
# CELL 5: BALANCED SAMPLING & EXPORT
# ==========================================
print(f"[{now()}] Starting balanced sampling...")

# Split back into attack and benign for sampling logic
df_attack_clean = df_combined[df_combined["label1"] == "attack"].copy()
df_benign_clean = df_combined[df_combined["label1"] == "benign"].copy()

# 1. Balance Attack Classes Perfectly
before_counts = df_attack_clean["label2"].value_counts().to_dict()
min_attack_samples = min(before_counts.values())

print(f"\nMinority attack class determines the baseline: {min_attack_samples} samples per attack type.")

def sample_attack_group(group_df):
    n = min(min_attack_samples, len(group_df))
    return group_df.sample(n=n, random_state=RANDOM_STATE)

grouped = df_attack_clean.groupby("label2", sort=False)
sampled_attacks = [sample_attack_group(grp) for name, grp in grouped]
df_attack_balanced = pd.concat(sampled_attacks, ignore_index=True, sort=False)
after_counts = df_attack_balanced["label2"].value_counts().to_dict()

# Print the visual report for attacks
report_counts(before_counts, after_counts, title="Balanced Attack Sampling Report (label2)")

# 2. Sample Benign Data
if 0 < BENIGN_FRAC < 1 and len(df_benign_clean) > 0:
    df_benign_sampled = df_benign_clean.sample(frac=BENIGN_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df_benign_sampled = df_benign_clean

print(f"Benign rows sampled: {len(df_benign_sampled):,} (Using frac={BENIGN_FRAC})")

# 3. Merge Final Dataset and Save
df_final = pd.concat([df_attack_balanced, df_benign_sampled], ignore_index=True, sort=False)

os.makedirs(OUT_DIR, exist_ok=True)
final_path = os.path.join(OUT_DIR, FINAL_OUT_FILE)
df_final.to_csv(final_path, index=False)

print(f"\n[{now()}] Success! Final dataset shape: {df_final.shape}")
print(f"[{now()}] Saved to: {final_path}")

[2026-03-11 13:53:56] Starting balanced sampling...

Minority attack class determines the baseline: 1868 samples per attack type.

Balanced Attack Sampling Report (label2)
Attack Type                    |       Before |        After
----------------------------------------------------------------
brute                          |         1868 |         1868
ddos                           |        36476 |         1868
malware                        |         7541 |         1868
mitm                           |         8062 |         1868
recon                          |        33648 |         1868
web                            |         2796 |         1868
----------------------------------------------------------------

Benign rows sampled: 13,680 (Using frac=0.1)

[2026-03-11 13:54:01] Success! Final dataset shape: (24888, 94)
[2026-03-11 13:54:01] Saved to: /content/drive/MyDrive/CIC_IIOT_2025/cic_iiot_2025_cleaned_and_balanced.csv
